# llm-memlab Kaggle Quickstart

This notebook keeps model paths configurable through environment variables and writes reports to `/kaggle/working`.

In [ ]:
!git clone https://github.com/thhanabun/LLMOptimization.git
%cd LLMOptimization
!python -m pip install -e .[torch,transformers]

In [ ]:
import os
os.environ.setdefault("LLM_MEMLAB_MODEL_ROOT", "/kaggle/working/hf_models")
os.environ.setdefault("LLM_MEMLAB_OUT_DIR", "/kaggle/working/llm_memlab_outputs")
os.environ.setdefault("LLM_MEMLAB_DEVICE", "auto")
os.environ.setdefault("LLM_MEMLAB_DTYPE", "auto")

Download a model from Hugging Face into `/kaggle/working`. For gated models, add an `HF_TOKEN` secret in Kaggle and uncomment the secret-loading block.

In [ ]:
# Optional for gated/private models:
# from kaggle_secrets import UserSecretsClient
# os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

from huggingface_hub import snapshot_download

HF_MODEL_ID = os.environ.get("HF_MODEL_ID", "hf-internal-testing/tiny-random-LlamaForCausalLM")
HF_TOKEN = os.environ.get("HF_TOKEN")
local_dir = os.path.join(os.environ["LLM_MEMLAB_MODEL_ROOT"], HF_MODEL_ID.replace("/", "__"))

model_path = snapshot_download(repo_id=HF_MODEL_ID, local_dir=local_dir, token=HF_TOKEN)
os.environ["LLM_MEMLAB_MODEL"] = model_path
print("Downloaded model to:", model_path)

In [ ]:
!python -m llm_memlab local-model-harness --root "$LLM_MEMLAB_MODEL_ROOT" --json-out /kaggle/working/local_model_fixtures.json
!python examples/cloud_smoke.py --tokens 1

In [ ]:
from IPython.display import HTML, display
dashboard = os.path.join(os.environ["LLM_MEMLAB_OUT_DIR"], "cloud_dashboard.html")
if os.path.exists(dashboard):
    display(HTML(open(dashboard, encoding="utf-8").read()))